In [1]:
import sys
import os
from collections import Counter
from pathlib import Path
from collections import defaultdict
import numpy as np
import pickle
import matplotlib.pyplot as plt

sys.path.append(os.path.join(os.getcwd(), ".."))
from models.experiment_model import StateSpaceModel

In [2]:
folder = r"..\data\constant_pump_speed_experiment"
files = os.listdir(folder)
trial_names = [f.split('_')[0] + '_' + f.split('_')[1] for f in files]
counts = Counter(trial_names)

pickle_files = sorted(Path(folder).glob("*.pickle"))
TOT_EXP = len(pickle_files)
TOT_EXP

35

In [3]:
def is_good_cv(x):
    if isinstance(x, str) or x == 0:
        return False

    try:
        v = float(x)
    except (TypeError, ValueError):
        return False

    return np.isfinite(v)

def clean(data, time_meausred): 
    cv_raw = data
    time_raw = time_meausred

    if len(cv_raw) != len(time_raw):
        raise ValueError("cv_list and time_list must have the same length")

    keep = [is_good_cv(cv) for cv in cv_raw]

    cv_clean = [cv for cv, k in zip(cv_raw, keep) if k]
    cv_clean = np.array(cv_clean, dtype=float)
    time_clean = [t  for t,  k in zip(time_raw, keep) if k]

    return cv_clean, time_clean

In [4]:
length_check = defaultdict(lambda: defaultdict(float))

for t in counts.keys():
    for i in range(counts[t]):
        data = rf"..\data\constant_pump_speed_experiment\{t}_{i}.pickle"
        with open(data, "rb") as f:
            data = pickle.load(f)
        
        cv_clean, time_clean = clean(data["cv"], data["time"])

        length_check["cv"][f"{t}_{i}"] = len(cv_clean)
        length_check["time"][f"{t}_{i}"] = len(time_clean)

if min(length_check["cv"].values()) != min(length_check["time"].values()):
    raise ValueError("cv_clean and time_clean has different length")

cut_to = min(length_check["cv"].values())
cut_to

436

In [5]:
speed, cv, time_data = np.empty((TOT_EXP), dtype=float), np.empty((TOT_EXP, cut_to), dtype=float), np.empty((TOT_EXP, cut_to), dtype=float)
j = 0

for t in counts.keys():
    temp_speed, temp_cv, temp_time = np.empty(counts[t], dtype=object), np.empty(counts[t], dtype=object), np.empty(counts[t], dtype=object)
    
    for i in range(counts[t]):
        data = rf"..\data\constant_pump_speed_experiment\{t}_{i}.pickle"

        with open(data, "rb") as f:
            data = pickle.load(f)
        
        pump_speed = data["pump_speed"]
        cv_raw = data["cv"]
        time_raw = data["time"]

        cv_clean, time_clean = clean(cv_raw, time_raw)

        cv_to_data, time_to_data = cv_clean[:cut_to], time_clean[:cut_to]
        temp_speed[i], temp_cv[i], temp_time[i] = np.array(pump_speed), np.array(cv_to_data), np.array(time_to_data)

        speed[j] = data["pump_speed"]
        cv[j,:] = cv_to_data
        time_data[j,:] = time_to_data
        j+=1

print(speed.shape, cv.shape, time_data.shape)

(35,) (35, 436) (35, 436)


In [6]:
model = StateSpaceModel(pred_x0=False)
A, B, C, D = model.multi_exp_param_estimation(cv=cv, time=time_data, u=speed)


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

This is Ipopt version 3.14.11, running with linear solver MUMPS 5.4.1.

Number of nonzeros in equality constraint Jacobian...:        0
Number of nonzeros in inequality constraint Jacobian.:        4
Number of nonzeros in Lagrangian Hessian.............:       10

Total number of variables............................:        4
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:        0
Total number of inequality c

In [7]:
A, B, C, D

(array([[-0.00893369,  0.        ],
        [ 0.02368513, -0.0089336 ]]),
 array([0.00151553, 0.        ]),
 array([0., 1.]),
 array(0.))

In [8]:
param = {
    "A": A,
    "B": B,
    "C": C,
    "D": D
}
with open(r"..\parameters\state_space_param.pickle", "wb") as f:
    pickle.dump(param, f)